## 9.3 — reshape ו-transpose

שני כלים לשינוי **צורת** מערך בלי לשנות את הערכים עצמם:

- **`a.reshape(new_shape)`** — מסדר מחדש את אותם ערכים לתוך צורה חדשה, בסדר קריאה קבוע (שורה-שורה). לא "מזיז" נתונים בין ריצות/זמנים — רק קורא להם אחרת.
- **`a.T`** (או `a.transpose()`) — **מחליף בין הצירים עצמם**: `(5, 60)` הופך ל-`(60, 5)`. בניגוד ל-reshape, זו לא רק "קריאה אחרת" — ציר 0 וציר 1 מחליפים תפקידים.

כבר ראינו `.T` בקצרה בסעיף 7.7 (על וקטורים); כאן מרחיבים למערך רב-ממדי.

In [ ]:
import numpy as np

g = 9.8
theta = np.radians(45)
v0_nom = 20.0
rng = np.random.default_rng(0)
n_runs, n_t = 5, 60
v0_runs = rng.normal(v0_nom, 1.0, size=n_runs)
t_flight_nom = 2 * v0_nom * np.sin(theta) / g
t = np.linspace(0, t_flight_nom, n_t)

Y = np.empty((n_runs, n_t))
for i in range(n_runs):
    Y[i] = v0_runs[i] * np.sin(theta) * t - 0.5 * g * t**2

### דוגמה: transpose לצורך תאימות לכלי חיצוני

נניח שכלי חיצוני (למשל טבלת pandas שנפגוש בשבוע 10) מצפה לקבל **שורה לכל נקודת זמן ועמודה לכל ריצה** — בדיוק ההפך מ-`Y`.

In [ ]:
Y_time_major = Y.T
print("Y.shape:", Y.shape, "  Y_time_major.shape:", Y_time_major.shape)

# בדיקת שפיות: הערך לא השתנה, רק מיקומו
print(Y[2, 10] == Y_time_major[10, 2])

### דוגמה: reshape לצורך שטוח (flatten)

לפעמים רוצים את כל 300 המדידות ($5\times 60$) כרשימה שטוחה אחת — למשל לחישוב היסטוגרמה כללית על כל המדידות ביחד (סעיף 8.8), בלי חלוקה לריצות.

In [ ]:
Y_flat = Y.reshape(-1)     # -1 אומר "תשלימו את הגודל לבד"
print(Y_flat.shape)           # (300,)

# אפשר לחזור לצורה המקורית:
Y_back = Y_flat.reshape(n_runs, n_t)
print(np.array_equal(Y, Y_back))

### באג נפוץ: `reshape` בהנחה שגויה על סדר הנתונים

`reshape` **לא** מודע למשמעות הפיזיקלית של הצירים — הוא רק קורא את הזיכרון בסדר קבוע ומחלק אותו לצורה החדשה. אם משנים `shape` בלי לוודא שהמשמעות עדיין נכונה (למשל, `reshape(n_t, n_runs)` על `Y` שצורתו `(n_runs, n_t)`, במקום `Y.T`), מקבלים מערך **באותה צורה שרצינו**, אבל עם ערכים **מעורבבים בין ריצות וזמנים** — שגיאה שקטה לגמרי.

In [ ]:
wrong = Y.reshape(n_t, n_runs)     # "עובד" בלי שגיאה, אבל הערכים מעורבבים!
right = Y.T                        # הדרך הנכונה להחליף צירים

print(np.array_equal(wrong, right))   # False - זה בדיוק הבאג

### נסו בעצמכם

הסבירו במילה או שתיים: למה `Y.reshape(n_t, n_runs)` ו-`Y.T` נותנים תוצאות שונות, למרות ששתיהן בצורה `(60, 5)`?

In [ ]:
# תשובה במילים (אין קוד להריץ כאן)

`````{admonition} פתרון
:class: dropdown, tip
`reshape` קורא את 300 הערכים בסדר הזיכרון הקבוע (כל הזמנים של ריצה 0, ואז כל הזמנים של ריצה 1, וכן הלאה) וממלא אותם לצורה החדשה באותו סדר — בלי קשר למשמעות הצירים. `Y.T` לעומת זאת שומר על ההתאמה הנכונה בין כל ערך לריצה ולזמן שלו, ורק מחליף את **סדר הצירים עצמו**.
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "מה ההבדל המהותי בין <code>a.reshape(...)</code> ל-<code>a.T</code>?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "reshape משנה את הערכים עצמם, T לא", "correct": False, "feedback": "לא — אף אחת מהן לא משנה ערכים, רק את סידורם."},
            {"answer": "T מחליף את תפקידי הצירים תוך שמירת ההתאמה הנכונה; reshape רק קורא מחדש את הזיכרון בסדר קבוע", "correct": True, "feedback": "נכון."},
            {"answer": "הן פונקציות זהות לחלוטין", "correct": False, "feedback": "לא נכון — ראינו דוגמה שבה הן נותנות תוצאות שונות."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

בנו מערך `Z` בצורה `(2, 3, 4)` כלשהו (ערכים לא חשובים — למשל `np.arange(24).reshape(2, 3, 4)`), והציגו את `Z.shape` אחרי `Z.transpose(1, 0, 2)`. מה קרה לכל ציר?

In [ ]:
# Z = np.arange(24).reshape(2, 3, 4)
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
Z = np.arange(24).reshape(2, 3, 4)
print(Z.shape)                       # (2, 3, 4)
print(Z.transpose(1, 0, 2).shape)    # (3, 2, 4)
```
`transpose(1, 0, 2)` אומר: "הציר שהיה במקום 1 יעבור למקום 0, הציר שהיה במקום 0 יעבור למקום 1, הציר שהיה במקום 2 נשאר במקומו" — כלומר הוחלפו רק שני הצירים הראשונים.
`````